This example demonstrates how to leverage the brushing/linked-selections support in the Vega pane to update a table.

In [ ]:
import altair as alt
import pandas as pd
import panel as pn

pn.extension('vega')

## Chart

The `brush` interval selection is named, so the `Vega` pane exposes it as `vega_pane.selection.param.brush`.

In [ ]:
df = pd.read_json("https://raw.githubusercontent.com/vega/vega/main/docs/data/penguins.json")

brush = alt.selection_interval(name='brush')

chart = alt.Chart(df).mark_point().encode(
    x=alt.X('Beak Length (mm):Q', scale=alt.Scale(zero=False)),
    y=alt.Y('Beak Depth (mm):Q', scale=alt.Scale(zero=False)),
    color=alt.condition(brush, 'Species:N', alt.value('lightgray'))
).properties(
    width=300,
    height=300
).add_params(
    brush
)

vega_pane = pn.ui.Vega(chart, debounce=10)

## Linked table

The table binds a pure filter function to the brush, so it only ever receives a new DataFrame.

In [ ]:
def filtered_table(selection):
    if not selection:
        return df.iloc[:0]
    query = ' & '.join(
        f'{crange[0]:.3f} <= `{col}` <= {crange[1]:.3f}'
        for col, crange in selection.items()
    )
    return df.query(query)

table = pn.ui.DataFrame(
    pn.bind(filtered_table, vega_pane.selection.param.brush),
    height=350,
    sizing_mode="stretch_width",
)

app = pn.ui.Column(
    pn.ui.Markdown('Select points on the plot and watch the linked table update.'),
    pn.ui.Row(vega_pane, table),
    sizing_mode='stretch_width',
)

page = pn.ui.Page(main=[app], title="Altair Brushing")

page

### Served App

In [ ]:
if pn.state.served:
    page.servable()